In [ ]:
import itertools
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [7]:
with open("mnist_small.pickle", "rb") as file:
    data = pickle.load(file)

In [8]:
_X = data["_X"].astype(np.float64)
_Y = data["_Y"].astype(np.int32)
print(_X.shape)
print(_X.dtype)
print(_Y.shape)
print(_Y.dtype)

(4000, 1, 28, 28)
float64
(4000, 1)
int32


In [14]:
# Split data into train and test sets
_X_train, _X_test, _Y_train, _Y_test = train_test_split(
    _X, _Y, test_size=0.2, random_state=42
)
print(_X_train.shape)
print(_Y_train.shape)
print(_X_test.shape)
print(_Y_test.shape)

(3200, 1, 28, 28)
(3200, 1)
(800, 1, 28, 28)
(800, 1)


In [ ]:
def scaler_transform(X):
    _mean = np.mean(X)
    _std = np.std(X)
    return (X - _mean) / _std


In [18]:
# Scale the data
# Note that we don't use SKLearn's StandardScaler here because our data is 4D and SKLearn's StandardScaler only works with 2D data.
# Therefore, we have to do it manually. We need to calculate mean and std of the training data.
_mean = np.mean(_X_train)
_std = np.std(_X_train)
print(f"Mean: {_mean}, Std: {_std}")

Mean: 33.14677574936224, Std: 78.34915146880222


In [ ]:
# Scale the X data
X_train = (_X_train - _mean) / _std
X_test = (_X_test - _mean) / _std

# Check the range of X before and after scaling
print("_X_train (before scaling):", _X_train.min(), "to", _X_train.max())
print("X_train (after scaling):", X_train.min(), "to", X_train.max())

# No scaling for labels
Y_train = _Y_train
Y_test = _Y_test


_X_train (before scaling): 0.0 to 255.0
X_train (after scaling): -0.4230648976787059 to 2.831597025514403
_X_test (before scaling): 0.0 to 255.0
X_test (after scaling): -0.4230648976787059 to 2.831597025514403


In [4]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc1 = nn.Linear(32 * 4 * 4, num_classes)

    def forward(self, X):
        X = self.conv1(X)
        X = self.relu(X)
        X = self.max_pool(X)
        X = self.conv2(X)
        X = self.relu(X)
        X = self.max_pool(X)
        X = self.adaptive_pool(X)
        X = X.view(X.shape[0], -1)
        X = self.fc1(X)
        return X


model = SimpleCNN(num_classes=10)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Function to get batches of indices
def get_batches_indices(num_samples, batch_size):
    indices = np.arange(num_samples)
    np.random.shuffle(indices)
    for start_idx in range(0, num_samples, batch_size):
        end_idx = min(start_idx + batch_size, num_samples)
        batch_indices = indices[start_idx:end_idx]
        yield batch_indices


# Example usage
num_samples = 20
batch_size = 6
for idxes in get_batches_indices(num_samples, batch_size):
    print(idxes)


[ 9 17 11  3 13  5]
[14 10 15  0  2  6]
[12 19 18  4  1  7]
[ 8 16]


In [ ]:
n_epochs = 100  # number of epochs to run
batch_size = 10  # size of each batch

for epoch in range(n_epochs):
    # Training Phase
    model.train()
    epoch_train_loss = 0.0
    epoch_train_f1 = 0.0
    logit_arr = []
    label_arr = []

    for idxes in get_batches_indices(len(X_train), batch_size):
        X_batch = X_train[idxes]
        Y_batch = Y_train[idxes]
        optimizer.zero_grad()
        Y_pred = model(X_batch)
        loss = loss_fn(Y_pred, Y_batch.view(-1))
        # Backward pass
        loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        # Update weights
        optimizer.step()
        # Multiplies the average loss per sample by the number of
        # samples in the batch to get the total loss for this batch.
        epoch_train_loss += loss.item() * len(idxes)
        logit_arr.append(Y_pred)
        label_arr.append(Y_batch)

    avg_train_loss = epoch_train_loss / len(X_train)

    logits = torch.concat(logit_arr, dim=0)
    labels = torch.concat(label_arr, dim=0)
    metrices, _, _ = calc_metrices(logits=logits, labels=labels.view(-1))
    avg_train_f1 = metrices["weighted avg"]["f1-score"]

    # Validation Phase
    if epoch % validation_interval == 0 or epoch == epoch_start:
        model.eval()
        val_loss = 0.0
        logit_arr = []
        label_arr = []
        with torch.no_grad():
            for X_val, Y_val in loader_val:
                Y_pred = model(X_val)
                val_loss += loss_fn(Y_pred, Y_val.view(-1)).item() * X_val.size(0)
                logit_arr.append(Y_pred)
                label_arr.append(Y_val)

        avg_val_loss = val_loss / len(loader_val.dataset)

        logits = torch.concat(logit_arr, dim=0)
        labels = torch.concat(label_arr, dim=0)
        metrices, _, _ = calc_metrices(logits=logits, labels=labels.view(-1))
        avg_val_f1 = metrices["weighted avg"]["f1-score"]

        scheduler.step(avg_val_loss)

        # Early Stopping and Checkpoint
        es = early_stopper(avg_val_loss)
        if es["best_loss"]:
            cph.save(
                save_path=save_path,
                model=model,
                optimizer=optimizer,
                val_loss=avg_val_loss,
                epoch=epoch,
            )
            print("Save model @ epoch:", epoch)
        if es["early_stop"]:
            print("Stopped at epoch:", epoch)
            break

    writer.add_scalars(
        log_name, {"loss/train": avg_train_loss, "loss/val": avg_val_loss}, epoch
    )
    writer.add_scalars(
        log_name, {"f1/train": avg_train_f1, "f1/val": avg_val_f1}, epoch
    )
